In [2]:
import os
import pandas as pd
from PIL import Image
import time
import moondream as md
from kaggle_secrets import UserSecretsClient

# ============================================================
# 1. Dataset paths
# ============================================================

pharma_dataset_path       = '/kaggle/input/datasets/salmankey/dataset/phrama_dataset/phrama_dataset'
pharma_csv_output_path    = '/kaggle/working/dataset_pharma_images.csv'
pharma_csv_output_with_ex = '/kaggle/working/dataset_pharma_with_extracted_details.csv'

aria_dataset_path         = '/kaggle/input/datasets/salmankey/dataset/dr_aria_dataset/dr_aria_dataset'
aria_csv_output_path      = '/kaggle/working/dataset_aria_images.csv'
aria_csv_output_with_ex   = '/kaggle/working/dataset_aria_with_extracted_details.csv'

MOONDREAM_MODEL = "moondream3.1-9B-A2B"

# ============================================================
# 2. Prompts
# ============================================================

PHARMA_EXTRACT_PROMPT = """
You are a prescription-reading assistant. Read the prescription image
and extract every detail that is printed on the document. Do not
diagnose, do not infer, and do not add anything that is not clearly
visible on the page.

Extract all of the following when they appear on the prescription:
- Drug name(s)
- Dose / strength
- Frequency (how often)
- Route (oral, topical, etc.)
- Duration
- Prescriber notes or instructions
- Any other text visible on the document (dates, clinic name,
  patient identifiers if shown)

Output format: write ONE short paragraph in simple, everyday English
that lists the extracted details in continuous prose. Do not use
headings, bullet points, numbered lists, labels, JSON, or tables.
If a detail is not visible on the document, do not invent it —
simply leave it out of the paragraph.
""".strip()

ARIA_EXTRACT_PROMPT = """
You are a medical-report reading assistant. Read the report image
and extract every detail that is printed on the document. Do not
diagnose, do not infer, and do not add anything that is not clearly
visible on the page.

Extract all of the following when they appear on the report:
- Patient name
- Age
- Gender
- Presenting complaint
- Diagnosis or impression
- Medication prescribed
- Follow-up recommendation
- Any other text visible on the document (dates, clinic name,
  clinician name if shown)

Output format: write ONE short paragraph in simple, everyday English
that lists the extracted details in continuous prose. Do not use
headings, bullet points, numbered lists, labels, JSON, or tables.
If a detail is not visible on the document, do not invent it —
simply leave it out of the paragraph.
""".strip()

# ============================================================
# 3. Scan helper
# ============================================================

IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.gif',
                    '.bmp', '.webp', '.tiff', '.tif')

def scan_images(folder):
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Dataset directory not found: {folder}")
    records = []
    for root, _, files in os.walk(folder):
        for file in sorted(files):
            if file.lower().endswith(IMAGE_EXTENSIONS):
                records.append({
                    'path': os.path.join(root, file),
                    'image_name': file,
                })
    return pd.DataFrame(records, columns=['path', 'image_name'])

# ============================================================
# 4. Retrieve secret and initialise Moondream
# ============================================================

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
MOONDREAM_API_KEY = user_secrets.get_secret("Moondream_testing")

if not MOONDREAM_API_KEY:
    raise ValueError(
        "The 'Moondream_testing' secret is empty. "
        "Go to Kaggle → Add-ons → Secrets, confirm the name is exactly "
        "'Moondream_testing', and restart the session."
    )

if 'moondream_client' not in globals():
    moondream_client = md.vl(api_key=MOONDREAM_API_KEY, model=MOONDREAM_MODEL)
    print(f"Moondream client initialised (model={MOONDREAM_MODEL}).")
else:
    print("Using existing moondream_client from a previous cell.")

# ============================================================
# 5. Extraction loop
# ============================================================

def extract_details_from_reports(df_in, prompt, label):
    extracted_details = []
    statuses = []

    print(f"\nProcessing {len(df_in)} {label} images with Moondream...")

    for index, row in df_in.iterrows():
        img_path = row['path']
        img_name = row['image_name']
        print(f"Processing ({index + 1}/{len(df_in)}): {img_name}")

        try:
            with Image.open(img_path) as img:
                image = img.convert("RGB")

            result = moondream_client.query(image=image, question=prompt)

            if isinstance(result, dict):
                answer = result.get('answer', '')
            else:
                answer = str(result)

            answer = str(answer).strip().strip('"').strip()
            if answer.lower().startswith('answer:'):
                answer = answer.split(':', 1)[1].strip()

            if not answer:
                answer = "No extracted details were returned by the model."

            extracted_details.append(answer)
            statuses.append('success' if answer else 'failed')

        except Exception as e:
            print(f"  Error on {img_name}: {e}")
            extracted_details.append("")
            statuses.append('failed')

        time.sleep(1)

    df_out = df_in.copy()
    df_out['extracted_detail']        = extracted_details
    df_out['extracted_detail_status'] = statuses
    return df_out

# ============================================================
# 6. PHARMA
# ============================================================

print("=" * 60)
print("PHARMA — prescription detail extraction")
print("=" * 60)

df_pharma = scan_images(pharma_dataset_path)
df_pharma.to_csv(pharma_csv_output_path, index=False)
print(f"Found {len(df_pharma)} prescription images.")

if len(df_pharma) == 0:
    raise ValueError("No prescription images found. Check pharma_dataset_path.")

df_pharma_out = extract_details_from_reports(
    df_pharma, PHARMA_EXTRACT_PROMPT, "pharma"
)
df_pharma_out.to_csv(pharma_csv_output_with_ex, index=False)
print("PHARMA extraction completed.")
print("Success:", (df_pharma_out['extracted_detail_status'] == 'success').sum())
print("Failed: ", (df_pharma_out['extracted_detail_status'] == 'failed').sum())
print(f"Saved: {pharma_csv_output_with_ex}")

# ============================================================
# 7. ARIA
# ============================================================

print("\n" + "=" * 60)
print("ARIA — report detail extraction")
print("=" * 60)

df_aria = scan_images(aria_dataset_path)
df_aria.to_csv(aria_csv_output_path, index=False)
print(f"Found {len(df_aria)} report images.")

if len(df_aria) == 0:
    raise ValueError("No report images found. Check aria_dataset_path.")

df_aria_out = extract_details_from_reports(
    df_aria, ARIA_EXTRACT_PROMPT, "aria"
)
df_aria_out.to_csv(aria_csv_output_with_ex, index=False)
print("ARIA extraction completed.")
print("Success:", (df_aria_out['extracted_detail_status'] == 'success').sum())
print("Failed: ", (df_aria_out['extracted_detail_status'] == 'failed').sum())
print(f"Saved: {aria_csv_output_with_ex}")

# ============================================================
# 8. Preview
# ============================================================

print("\n===== PHARMA preview =====")
display(df_pharma_out.head())

print("\n===== ARIA preview =====")
display(df_aria_out.head())

Using existing moondream_client from a previous cell.
PHARMA — prescription detail extraction
Found 10 prescription images.

Processing 10 pharma images with Moondream...
Processing (1/10): Medical_Report_2026-02-12.png
Processing (2/10): Medical_Report_2026-02-15 (3).png
Processing (3/10): Medical_Report_2026-02-15 (5).png
Processing (4/10): Medical_Report_2026-02-16 (10).png
Processing (5/10): Medical_Report_2026-02-16 (11).png
Processing (6/10): Medical_Report_2026-02-16 (12).png
Processing (7/10): Medical_Report_2026-02-16 (9).png
Processing (8/10): Medical_Report_2026-02-18.png
Processing (9/10): Medical_Report_2026-02-20.png
Processing (10/10): Medical_Report_Dr_Zavian_2026-10-24.png
PHARMA extraction completed.
Success: 10
Failed:  0
Saved: /kaggle/working/dataset_pharma_with_extracted_details.csv

ARIA — report detail extraction
Found 10 report images.

Processing 10 aria images with Moondream...
Processing (1/10): Medical_Report_2026-02-12 (1).png
Processing (2/10): Medical_Re

,path,image_name,extracted_detail,extracted_detail_status
0,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-12.png,"{\n ""Drug Name(s)"": [\n ""Adapalene"",\n ...",success
1,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-15 (3).png,"{\n ""drug_name"": [\n ""Benzoyl Peroxide 2.5...",success
2,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-15 (5).png,"{\n ""medications"": [\n {\n ""drug_name...",success
3,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-16 (10).png,"{\n ""Drug Name(s)"": [\n ""Benzoyl Peroxide ...",success
4,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-16 (11).png,"{\n ""drug_name"": [\n ""BENZYL PEROXIDE"",\n ...",success



===== ARIA preview =====


,path,image_name,extracted_detail,extracted_detail_status
0,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-12 (1).png,Patient name: Mohammehd Saliman\nAge: 22\nGend...,success
1,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-15 (2).png,Patient name: SALMAN\nAge: 22\nSex: male\nPres...,success
2,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-15 (3).png,Patient name: Salman\nAge: 22\nGender: Male\nP...,success
3,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-15 (4).png,Patient name: SALMAN\nAge: 22\nSex: male\nPres...,success
4,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-15.png,Salman Salman is 20 years old male. His presen...,success


In [ ]:
pip install moondream

In [3]:
import os
import json
import re
import time
import pandas as pd
from PIL import Image
import moondream as md
from kaggle_secrets import UserSecretsClient

# ============================================================
# 1. Dataset paths
# ============================================================

pharma_dataset_path       = '/kaggle/input/datasets/salmankey/dataset/phrama_dataset/phrama_dataset'
pharma_csv_output_path    = '/kaggle/working/mednexus_pharma_image_index.csv'
pharma_csv_output_with_ex = '/kaggle/working/mednexus_pharma_extracted_details.csv'

aria_dataset_path         = '/kaggle/input/datasets/salmankey/dataset/dr_aria_dataset/dr_aria_dataset'
aria_csv_output_path      = '/kaggle/working/mednexus_aria_image_index.csv'
aria_csv_output_with_ex   = '/kaggle/working/mednexus_aria_extracted_details.csv'

MOONDREAM_MODEL = "moondream3.1-9B-A2B"

# ============================================================
# 2. Prompts — free-form paragraph, not a fixed template
# ============================================================

PHARMA_EXTRACT_PROMPT = """
You are reading a prescription image. Write ONE short paragraph in
plain English that describes everything visible on the prescription.

The paragraph should read like a short narrative, for example:

"The prescription lists Amlodipine 10 mg, taken once daily by mouth
for 30 days, with a note to take it in the morning. The clinic name
and date appear at the top of the page."

Do not use the example above as a template to copy — it is only to
show you the shape of the output (one paragraph, plain sentences, no
labels).

Content to include, when visible on the prescription:
- Drug name and strength
- Dose and frequency
- Route of administration
- Duration
- Prescriber notes or instructions
- Dates, clinic name, patient identifiers, and any other text

Rules:
- The output must be a SINGLE paragraph of continuous prose.
- Do not use labels such as "Drug name:" or "Dose:".
- Do not use line breaks, bullet points, numbered lists, JSON, tables,
  or key-value pairs.
- Do not repeat the paragraph. Write it once.
- Copy values as printed. Do not invent details that are not visible.
- If a detail is not visible, leave it out of the paragraph.
""".strip()


ARIA_EXTRACT_PROMPT = """
You are reading a medical report image. Write ONE short paragraph in
plain English that describes everything visible on the report.

The paragraph should read like a short narrative, for example:

"The report is for John Doe, a 34-year-old female presenting with a
persistent low mood. The diagnosis is a moderate depressive episode,
and Sertraline 50 mg has been prescribed, with a follow-up review
recommended in four weeks."

Do not use the example above as a template to copy — it is only to
show you the shape of the output (one paragraph, plain sentences, no
labels).

Content to include, when visible on the report:
- Patient name, age, and gender
- Presenting complaint
- Diagnosis or impression
- Medication prescribed (with dose if shown)
- Follow-up recommendation
- Dates, clinic name, clinician name, and any other text

Rules:
- The output must be a SINGLE paragraph of continuous prose.
- Do not use labels such as "Patient name:" or "Diagnosis:".
- Do not use line breaks, bullet points, numbered lists, JSON, tables,
  or key-value pairs.
- Do not repeat the paragraph. Write it once.
- Copy values as printed. Do not invent details that are not visible.
- If a detail is not visible, leave it out of the paragraph.
- The report may come from a different specialty (ENT, dermatology,
  pharmacy, mental health). Describe what is on the page; do not
  assume the specialty.
""".strip()

# ============================================================
# 3. Scan helper
# ============================================================

IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.gif',
                    '.bmp', '.webp', '.tiff', '.tif')

def scan_images(folder):
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Dataset directory not found: {folder}")
    records = []
    for root, _, files in os.walk(folder):
        for file in sorted(files):
            if file.lower().endswith(IMAGE_EXTENSIONS):
                records.append({
                    'path': os.path.join(root, file),
                    'image_name': file,
                })
    return pd.DataFrame(records, columns=['path', 'image_name'])

# ============================================================
# 4. Retrieve secret and initialise Moondream
# ============================================================

user_secrets = UserSecretsClient()
MOONDREAM_API_KEY = user_secrets.get_secret("Moondream_testing")

if not MOONDREAM_API_KEY:
    raise ValueError(
        "The 'Moondream_testing' secret is empty. "
        "Go to Kaggle → Add-ons → Secrets, confirm the name is exactly "
        "'Moondream_testing', and restart the session."
    )

if 'moondream_client' not in globals():
    moondream_client = md.vl(api_key=MOONDREAM_API_KEY, model=MOONDREAM_MODEL)
    print(f"Moondream client initialised (model={MOONDREAM_MODEL}).")
else:
    print("Using existing moondream_client from a previous cell.")

# ============================================================
# 5. Post-processing helpers
# ============================================================

def _json_to_paragraph(text):
    """
    If Moondream returns JSON despite the instruction, convert the
    top-level key/value pairs into a plain-English paragraph so that
    the 'extracted_detail' column is always paragraph-shaped.
    """
    stripped = text.strip()
    if not stripped.startswith("{"):
        return text

    try:
        obj = json.loads(stripped)
    except Exception:
        cleaned = re.sub(r'[{}"]', ' ', stripped)
        cleaned = re.sub(r'\s+', ' ', cleaned).strip()
        return cleaned

    parts = []
    for key, value in obj.items():
        pretty_key = key.replace("_", " ").strip().capitalize()
        if isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    inner = ", ".join(f"{k.replace('_', ' ')}: {v}"
                                      for k, v in item.items() if v)
                    parts.append(f"{pretty_key}: {inner}")
                else:
                    parts.append(f"{pretty_key}: {item}")
        elif value:
            parts.append(f"{pretty_key}: {value}")

    return ". ".join(parts) + "."


def _de_duplicate_paragraph(text):
    """
    If the model repeated the extraction twice in one response, keep
    only the first paragraph. Guards against the duplication failure
    mode observed on form-like documents.
    """
    blocks = [b.strip() for b in re.split(r'\n\s*\n', text) if b.strip()]
    if len(blocks) <= 1:
        return text.strip()
    return blocks[0]

# ============================================================
# 6. Extraction loop
# ============================================================

def extract_details_from_reports(df_in, prompt, label):
    extracted_details = []
    statuses = []

    print(f"\nProcessing {len(df_in)} {label} images with Moondream...")

    for index, row in df_in.iterrows():
        img_path = row['path']
        img_name = row['image_name']
        print(f"Processing ({index + 1}/{len(df_in)}): {img_name}")

        try:
            with Image.open(img_path) as img:
                image = img.convert("RGB")

            try:
                result = moondream_client.query(
                    image=image,
                    question=prompt,
                    max_tokens=200,
                )
            except TypeError:
                result = moondream_client.query(image=image, question=prompt)

            if isinstance(result, dict):
                answer = result.get('answer', '')
            else:
                answer = str(result)

            answer = str(answer).strip().strip('"').strip()
            if answer.lower().startswith('answer:'):
                answer = answer.split(':', 1)[1].strip()

            answer = _json_to_paragraph(answer)
            answer = _de_duplicate_paragraph(answer)

            if not answer:
                answer = "No extracted details were returned by the model."

            extracted_details.append(answer)
            statuses.append('success' if answer else 'failed')

        except Exception as e:
            print(f"  Error on {img_name}: {e}")
            extracted_details.append("")
            statuses.append('failed')

        time.sleep(1)

    df_out = df_in.copy()
    df_out['extracted_detail']        = extracted_details
    df_out['extracted_detail_status'] = statuses
    return df_out

# ============================================================
# 7. PHARMA
# ============================================================

print("=" * 60)
print("PHARMA — prescription detail extraction")
print("=" * 60)

df_pharma = scan_images(pharma_dataset_path)
df_pharma.to_csv(pharma_csv_output_path, index=False)
print(f"Found {len(df_pharma)} prescription images.")

if len(df_pharma) == 0:
    raise ValueError("No prescription images found. Check pharma_dataset_path.")

df_pharma_out = extract_details_from_reports(
    df_pharma, PHARMA_EXTRACT_PROMPT, "pharma"
)
df_pharma_out.to_csv(pharma_csv_output_with_ex, index=False)
print("PHARMA extraction completed.")
print("Success:", (df_pharma_out['extracted_detail_status'] == 'success').sum())
print("Failed: ", (df_pharma_out['extracted_detail_status'] == 'failed').sum())
print(f"Saved: {pharma_csv_output_with_ex}")

# ============================================================
# 8. ARIA
# ============================================================

print("\n" + "=" * 60)
print("ARIA — report detail extraction")
print("=" * 60)

df_aria = scan_images(aria_dataset_path)
df_aria.to_csv(aria_csv_output_path, index=False)
print(f"Found {len(df_aria)} report images.")

if len(df_aria) == 0:
    raise ValueError("No report images found. Check aria_dataset_path.")

df_aria_out = extract_details_from_reports(
    df_aria, ARIA_EXTRACT_PROMPT, "aria"
)
df_aria_out.to_csv(aria_csv_output_with_ex, index=False)
print("ARIA extraction completed.")
print("Success:", (df_aria_out['extracted_detail_status'] == 'success').sum())
print("Failed: ", (df_aria_out['extracted_detail_status'] == 'failed').sum())
print(f"Saved: {aria_csv_output_with_ex}")

# ============================================================
# 9. Preview
# ============================================================

print("\n===== PHARMA preview =====")
display(df_pharma_out.head())

print("\n===== ARIA preview =====")
display(df_aria_out.head())

Using existing moondream_client from a previous cell.
PHARMA — prescription detail extraction
Found 10 prescription images.

Processing 10 pharma images with Moondream...
Processing (1/10): Medical_Report_2026-02-12.png
Processing (2/10): Medical_Report_2026-02-15 (3).png
Processing (3/10): Medical_Report_2026-02-15 (5).png
Processing (4/10): Medical_Report_2026-02-16 (10).png
Processing (5/10): Medical_Report_2026-02-16 (11).png
Processing (6/10): Medical_Report_2026-02-16 (12).png
Processing (7/10): Medical_Report_2026-02-16 (9).png
Processing (8/10): Medical_Report_2026-02-18.png
Processing (9/10): Medical_Report_2026-02-20.png
Processing (10/10): Medical_Report_Dr_Zavian_2026-10-24.png
PHARMA extraction completed.
Success: 10
Failed:  0
Saved: /kaggle/working/mednexus_pharma_extracted_details.csv

ARIA — report detail extraction
Found 10 report images.

Processing 10 aria images with Moondream...
Processing (1/10): Medical_Report_2026-02-12 (1).png
Processing (2/10): Medical_Report

,path,image_name,extracted_detail,extracted_detail_status
0,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-12.png,"The prescription lists Amlodipine 10 mg, taken...",success
1,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-15 (3).png,"The prescription lists Amlodipine 10 mg, taken...",success
2,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-15 (5).png,"The prescription lists Amlodipine 10 mg, taken...",success
3,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-16 (10).png,"The prescription lists Amlodipine 10 mg, taken...",success
4,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-16 (11).png,"The prescription lists Amlodipine 10 mg, taken...",success



===== ARIA preview =====


,path,image_name,extracted_detail,extracted_detail_status
0,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-12 (1).png,"The report is for Mohammehd Saliman, a 22-year...",success
1,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-15 (2).png,"The report is for Salman, a 22-year-old male p...",success
2,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-15 (3).png,"The report is for Salman, a 22-year-old male p...",success
3,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-15 (4).png,"The report is for Salman, 22 years old male, p...",success
4,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-15.png,"Salman Salman, 20, male, is seen at ENT Specia...",success


In [4]:
import pandas as pd

files = [
    '/kaggle/working/mednexus_pharma_extracted_details.csv',
    '/kaggle/working/mednexus_aria_extracted_details.csv',
]

for f in files:
    df = pd.read_csv(f)

    # Rename extracted_detail -> ground_truth
    df = df.rename(columns={'extracted_detail': 'ground_truth'})

    # Keep only path, image_name, ground_truth
    df = df[['path', 'image_name', 'ground_truth']]

    # Save back
    df.to_csv(f, index=False)

    print(f"Updated: {f}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Rows:    {len(df)}")

Updated: /kaggle/working/mednexus_pharma_extracted_details.csv
  Columns: ['path', 'image_name', 'ground_truth']
  Rows:    10
Updated: /kaggle/working/mednexus_aria_extracted_details.csv
  Columns: ['path', 'image_name', 'ground_truth']
  Rows:    10


In [5]:

data=pd.read_csv("/kaggle/working/mednexus_pharma_extracted_details.csv")
data

,path,image_name,ground_truth
0,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-12.png,"The prescription lists Amlodipine 10 mg, taken..."
1,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-15 (3).png,"The prescription lists Amlodipine 10 mg, taken..."
2,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-15 (5).png,"The prescription lists Amlodipine 10 mg, taken..."
3,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-16 (10).png,"The prescription lists Amlodipine 10 mg, taken..."
4,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-16 (11).png,"The prescription lists Amlodipine 10 mg, taken..."
5,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-16 (12).png,The prescription lists Benzoyl Peroxide 2.5% G...
6,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-16 (9).png,"The prescription lists Amlodipine 10 mg, taken..."
7,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-18.png,"The prescription lists Amlodipine 10 mg, taken..."
8,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-20.png,The prescription lists Benzoyl Peroxide 2.5% G...
9,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_Dr_Zavian_2026-10-24.png,"The prescription lists Amlodipine 10 mg, taken..."


In [9]:
# ============================================================
# Pharma and Aria: add 'extracted_detail' next to existing 'ground_truth'
# Input files already have: path, image_name, ground_truth
# ============================================================

import os
import json
import re
import time
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import moondream as md
from kaggle_secrets import UserSecretsClient

# ------------------------------------------------------------
# Configuration — point at the files you already have
# ------------------------------------------------------------

PHARMA_INPUT_CSV  = '/kaggle/input/datasets/salmankey/mednexus-pharma-data/mednexus_pharma_data.csv'
PHARMA_IMAGE_DIR  = '/kaggle/input/datasets/salmankey/dataset/phrama_dataset/phrama_dataset'
PHARMA_OUTPUT_CSV = '/kaggle/working/mednexus_pharma_extracted_details2.csv'

ARIA_INPUT_CSV    = '/kaggle/input/datasets/salmankey/mednexus-aria-data/mednexus_aria_data.csv'
ARIA_IMAGE_DIR    = '/kaggle/input/datasets/salmankey/dataset/dr_aria_dataset/dr_aria_dataset'
ARIA_OUTPUT_CSV   = '/kaggle/working/mednexus_aria_extracted_details2.csv'

MOONDREAM_MODEL = "moondream3-preview"

# ------------------------------------------------------------
# Prompts
# ------------------------------------------------------------

PHARMA_EXTRACT_PROMPT = """
You are reading a prescription image. Write ONE short paragraph in
plain English that describes everything visible on the prescription.

The paragraph should read like a short narrative, for example:

"The prescription lists Amlodipine 10 mg, taken once daily by mouth
for 30 days, with a note to take it in the morning. The clinic name
and date appear at the top of the page."

Do not use the example above as a template to copy — it is only to
show you the shape of the output (one paragraph, plain sentences, no
labels).

Content to include, when visible on the prescription:
- Drug name and strength
- Dose and frequency
- Route of administration
- Duration
- Prescriber notes or instructions
- Dates, clinic name, patient identifiers, and any other text

Rules:
- The output must be a SINGLE paragraph of continuous prose.
- Do not use labels such as "Drug name:" or "Dose:".
- Do not use line breaks, bullet points, numbered lists, JSON, tables,
  or key-value pairs.
- Do not repeat the paragraph. Write it once.
- Copy values as printed. Do not invent details that are not visible.
- If a detail is not visible, leave it out of the paragraph.
""".strip()


ARIA_EXTRACT_PROMPT = """
You are reading a medical report image. Write ONE short paragraph in
plain English that describes everything visible on the report.

The paragraph should read like a short narrative, for example:

"The report is for John Doe, a 34-year-old female presenting with a
persistent low mood. The diagnosis is a moderate depressive episode,
and Sertraline 50 mg has been prescribed, with a follow-up review
recommended in four weeks."

Do not use the example above as a template to copy — it is only to
show you the shape of the output (one paragraph, plain sentences, no
labels).

Content to include, when visible on the report:
- Patient name, age, and gender
- Presenting complaint
- Diagnosis or impression
- Medication prescribed (with dose if shown)
- Follow-up recommendation
- Dates, clinic name, clinician name, and any other text

Rules:
- The output must be a SINGLE paragraph of continuous prose.
- Do not use labels such as "Patient name:" or "Diagnosis:".
- Do not use line breaks, bullet points, numbered lists, JSON, tables,
  or key-value pairs.
- Do not repeat the paragraph. Write it once.
- Copy values as printed. Do not invent details that are not visible.
- If a detail is not visible, leave it out of the paragraph.
- The report may come from a different specialty (ENT, dermatology,
  pharmacy, mental health). Describe what is on the page; do not
  assume the specialty.
""".strip()

# ------------------------------------------------------------
# Secrets and client
# ------------------------------------------------------------

user_secrets = UserSecretsClient()
MOONDREAM_API_KEY = user_secrets.get_secret("Moondream_testing")
if not MOONDREAM_API_KEY:
    raise ValueError("Moondream_testing secret is empty or missing.")

client = md.vl(api_key=MOONDREAM_API_KEY, model=MOONDREAM_MODEL)

# ------------------------------------------------------------
# Guard helpers
# ------------------------------------------------------------

def _json_to_paragraph(text):
    stripped = text.strip()
    if not stripped.startswith("{"):
        return text
    try:
        obj = json.loads(stripped)
    except Exception:
        cleaned = re.sub(r'[{}"]', ' ', stripped)
        return re.sub(r'\s+', ' ', cleaned).strip()
    parts = []
    for k, v in obj.items():
        pretty = k.replace("_", " ").strip().capitalize()
        if isinstance(v, list):
            for item in v:
                if isinstance(item, dict):
                    inner = ", ".join(f"{kk.replace('_', ' ')}: {vv}"
                                      for kk, vv in item.items() if vv)
                    parts.append(f"{pretty}: {inner}")
                else:
                    parts.append(f"{pretty}: {item}")
        elif v:
            parts.append(f"{pretty}: {v}")
    return ". ".join(parts) + "."


def _de_duplicate_paragraph(text):
    blocks = [b.strip() for b in re.split(r'\n\s*\n', text) if b.strip()]
    return blocks[0] if len(blocks) > 1 else text.strip()

# ------------------------------------------------------------
# Core loop
# ------------------------------------------------------------

def add_extracted_column(input_csv, image_dir, output_csv, prompt, label):
    print("=" * 60)
    print(f"{label.upper()} — adding extracted_detail column")
    print("=" * 60)

    df = pd.read_csv(input_csv)
    print(f"Loaded {len(df)} rows from {input_csv}")
    print(f"Existing columns: {list(df.columns)}")

    if 'ground_truth' not in df.columns:
        raise ValueError(
            f"{input_csv} must have a 'ground_truth' column. "
            f"Found: {list(df.columns)}"
        )

    # Rebuild 'path' to point at the actual Kaggle image directory
    df['path'] = df['image_name'].apply(
        lambda name: os.path.join(image_dir, str(name))
    )

    # Verify images exist
    df['_exists'] = df['path'].apply(os.path.isfile)
    n_missing = int((~df['_exists']).sum())
    if n_missing > 0:
        print(f"\n⚠️  {n_missing} image(s) not found:")
        print(df.loc[~df['_exists'], ['image_name', 'path']].to_string(index=False))
        df = df[df['_exists']].reset_index(drop=True)
        print(f"Continuing with {len(df)} existing images.")
    df = df.drop(columns=['_exists'])

    if len(df) == 0:
        raise ValueError(f"No valid images for {label}.")

    # Run Moondream
    extracted = []
    statuses  = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=label):
        try:
            with Image.open(row['path']) as img:
                image = img.convert("RGB")

            try:
                result = client.query(image=image, question=prompt, max_tokens=200)
            except TypeError:
                result = client.query(image=image, question=prompt)

            ans = result.get('answer', '') if isinstance(result, dict) else str(result)
            ans = str(ans).strip().strip('"').strip()
            if ans.lower().startswith('answer:'):
                ans = ans.split(':', 1)[1].strip()

            if ans.startswith("{"):
                ans = _json_to_paragraph(ans)
            ans = _de_duplicate_paragraph(ans)

            extracted.append(ans)
            statuses.append('success' if ans else 'failed')
        except Exception as e:
            print(f"\nError on {row['image_name']}: {e}")
            extracted.append("")
            statuses.append('failed')
        time.sleep(1)

    # Add the new columns; keep the existing 'ground_truth'
    df['extracted_detail']        = extracted
    df['extracted_detail_status'] = statuses

    # Reorder so ground_truth and extracted_detail sit side by side
    df = df[['path', 'image_name', 'ground_truth', 'extracted_detail', 'extracted_detail_status']]

    df.to_csv(output_csv, index=False)
    print(f"\nSaved: {output_csv}")
    print(f"Successful: {(df['extracted_detail_status'] == 'success').sum()}/{len(df)}")
    print(f"Output columns: {list(df.columns)}")

    print("\nFirst row preview:")
    print(f"  image_name:       {df.iloc[0]['image_name']}")
    print(f"  ground_truth:     {str(df.iloc[0]['ground_truth'])[:80]}...")
    print(f"  extracted_detail: {str(df.iloc[0]['extracted_detail'])[:80]}...")
    return df

# ------------------------------------------------------------
# Run both
# ------------------------------------------------------------

df_pharma = add_extracted_column(
    input_csv  = PHARMA_INPUT_CSV,
    image_dir  = PHARMA_IMAGE_DIR,
    output_csv = PHARMA_OUTPUT_CSV,
    prompt     = PHARMA_EXTRACT_PROMPT,
    label      = "pharma",
)

df_aria = add_extracted_column(
    input_csv  = ARIA_INPUT_CSV,
    image_dir  = ARIA_IMAGE_DIR,
    output_csv = ARIA_OUTPUT_CSV,
    prompt     = ARIA_EXTRACT_PROMPT,
    label      = "aria",
)

print("\n===== PHARMA preview =====")
display(df_pharma.head())

print("\n===== ARIA preview =====")
display(df_aria.head())

PHARMA — adding extracted_detail column
Loaded 10 rows from /kaggle/input/datasets/salmankey/mednexus-pharma-data/mednexus_pharma_data.csv
Existing columns: ['path', 'image_name', 'ground_truth', 'extracted_detail_status']


pharma:   0%|          | 0/10 [00:00<?, ?it/s]


Saved: /kaggle/working/mednexus_pharma_extracted_details2.csv
Successful: 10/10
Output columns: ['path', 'image_name', 'ground_truth', 'extracted_detail', 'extracted_detail_status']

First row preview:
  image_name:       Medical_Report_2026-02-12.png
  ground_truth:     A 22-year-old male came in with persistent blackheads (comedonal acne). There ar...
  extracted_detail: The prescription lists Amlodipine 10 mg, taken once daily by mouth for 30 days, ...
ARIA — adding extracted_detail column
Loaded 10 rows from /kaggle/input/datasets/salmankey/mednexus-aria-data/mednexus_aria_data.csv
Existing columns: ['path', 'image_name', 'ground_truth', 'extracted_detail_status']


aria:   0%|          | 0/10 [00:00<?, ?it/s]


Saved: /kaggle/working/mednexus_aria_extracted_details2.csv
Successful: 10/10
Output columns: ['path', 'image_name', 'ground_truth', 'extracted_detail', 'extracted_detail_status']

First row preview:
  image_name:       Medical_Report_2026-02-12 (1).png
  ground_truth:     This report concerns Mohammehd Saliman, a 22-year-old male....
  extracted_detail: The report is for Mohammede Saliman, a 22-year-old male presenting with a persis...

===== PHARMA preview =====


,path,image_name,ground_truth,extracted_detail,extracted_detail_status
0,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-12.png,A 22-year-old male came in with persistent bla...,"The prescription lists Amlodipine 10 mg, taken...",success
1,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-15 (3).png,"Salman, a 22-year-old male from Buner, was see...",The prescription lists Benzoyl Peroxide 2.5% G...,success
2,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-15 (5).png,"This prescription, dated February 15, 2026, is...",The prescription lists Benzoyl Peroxide 2.5% G...,success
3,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-16 (10).png,"Salman, a 22-year-old male from Buner (patient...","The prescription lists Amlodipine 10 mg, taken...",success
4,/kaggle/input/datasets/salmankey/dataset/phram...,Medical_Report_2026-02-16 (11).png,"On February 16, 2026, Dr. Zavian, Consultant D...","The prescription lists Amlodipine 10 mg, taken...",success



===== ARIA preview =====


,path,image_name,ground_truth,extracted_detail,extracted_detail_status
0,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-12 (1).png,"This report concerns Mohammehd Saliman, a 22-y...","The report is for Mohammede Saliman, a 22-year...",success
1,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-15 (2).png,"Salman, a 22-year-old male, is the patient nam...","The report is for Salman, a 22-year-old male p...",success
2,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-15 (3).png,"Salman, a 22-year-old male, has been prescribe...","The report is for Salman, a 22-year-old male p...",success
3,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-15 (4).png,"Salman, 22, male, came in with a fever that ha...","The report is for Salman Salman, a 22-year-old...",success
4,/kaggle/input/datasets/salmankey/dataset/dr_ar...,Medical_Report_2026-02-15.png,"At the ENT Specialist Clinic, Consultation & S...","The report is for Salman Salman, a 20-year-old...",success


In [10]:
# ============================================================
# Step 2: Dual-Model Cosine Similarity — GT vs Extracted
# Pharma and Aria
# ============================================================

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

PHARMA_INPUT_CSV   = '/kaggle/working/mednexus_pharma_extracted_details2.csv'
PHARMA_OUTPUT_CSV  = '/kaggle/working/mednexus_pharma_cosine_eval_dual.csv'
PHARMA_SUMMARY_CSV = '/kaggle/working/mednexus_pharma_cosine_eval_dual_summary.csv'

ARIA_INPUT_CSV     = '/kaggle/working/mednexus_aria_extracted_details2.csv'
ARIA_OUTPUT_CSV    = '/kaggle/working/mednexus_aria_cosine_eval_dual.csv'
ARIA_SUMMARY_CSV   = '/kaggle/working/mednexus_aria_cosine_eval_dual_summary.csv'

GENERAL_MODEL   = 'BAAI/bge-large-en-v1.5'
MEDICAL_MODEL   = 'abhinand/MedEmbed-base-v0.1'
MATCH_THRESHOLD = 0.75

GT_COLUMN  = 'ground_truth'
GEN_COLUMN = 'extracted_detail'

# ------------------------------------------------------------
# Core function — run for one specialist
# ------------------------------------------------------------

def run_cosine_eval(input_csv, output_csv, summary_csv, label):
    print("=" * 60)
    print(f"{label.upper()} — dual-model cosine similarity")
    print("=" * 60)

    df = pd.read_csv(input_csv)
    print(f"Loaded {len(df)} rows from {input_csv}")

    # Filter rows where both columns are non-empty
    df = df[
        df[GEN_COLUMN].astype(str).str.strip().ne('') &
        df[GEN_COLUMN].notna() &
        df[GT_COLUMN].astype(str).str.strip().ne('') &
        df[GT_COLUMN].notna()
    ].copy().reset_index(drop=True)
    print(f"Valid rows: {len(df)}")

    if len(df) == 0:
        print(f"No valid rows for {label}; skipping.")
        return None

    df['gt_words']  = df[GT_COLUMN].apply(lambda s: len(str(s).split()))
    df['gen_words'] = df[GEN_COLUMN].apply(lambda s: len(str(s).split()))

    # Score with a given model
    def score_model(model_name, tag):
        print(f"\nLoading {model_name} ...")
        model = SentenceTransformer(model_name)

        gt_emb = model.encode(
            df[GT_COLUMN].tolist(),
            convert_to_tensor=True, normalize_embeddings=True,
        )
        gen_emb = model.encode(
            df[GEN_COLUMN].tolist(),
            convert_to_tensor=True, normalize_embeddings=True,
        )
        sims = util.cos_sim(gt_emb, gen_emb).diagonal().cpu().numpy()

        df[f'sim_{tag}'] = sims
        print(f"  mean={sims.mean():.4f}  std={sims.std():.4f}  "
              f"min={sims.min():.4f}  max={sims.max():.4f}  "
              f"match@0.75={(sims >= MATCH_THRESHOLD).sum()}/{len(sims)}")
        return sims

    sims_bge = score_model(GENERAL_MODEL, 'bge')
    sims_med = score_model(MEDICAL_MODEL, 'medembed')

    # Per-image comparison
    df['diff_med_minus_bge'] = df['sim_medembed'] - df['sim_bge']
    df['winner'] = df.apply(
        lambda r: 'MedEmbed' if r['sim_medembed'] > r['sim_bge']
        else ('BGE' if r['sim_bge'] > r['sim_medembed'] else 'Tie'),
        axis=1
    )

    print(f"\n===== {label.upper()} PER-IMAGE COMPARISON =====")
    preview = ['image_name', 'sim_bge', 'sim_medembed',
               'diff_med_minus_bge', 'winner']
    print(df[preview].sort_values('diff_med_minus_bge', ascending=False).to_string(index=False))

    # Head-to-head verdict
    print(f"\n===== {label.upper()} HEAD-TO-HEAD VERDICT =====")

    bge_mean = df['sim_bge'].mean()
    med_mean = df['sim_medembed'].mean()
    print(f"Mean similarity:     BGE={bge_mean:.4f}  vs  MedEmbed={med_mean:.4f}")
    print(f"  → Higher mean:      {'MedEmbed' if med_mean > bge_mean else 'BGE'}")

    bge_match = int((df['sim_bge'] >= MATCH_THRESHOLD).sum())
    med_match = int((df['sim_medembed'] >= MATCH_THRESHOLD).sum())
    print(f"Match rate @{MATCH_THRESHOLD}:  BGE={bge_match}/{len(df)}  vs  MedEmbed={med_match}/{len(df)}")
    print(f"  → Higher match rate: {'MedEmbed' if med_match > bge_match else ('BGE' if bge_match > med_match else 'Tie')}")

    n_med_wins = int((df['winner'] == 'MedEmbed').sum())
    n_bge_wins = int((df['winner'] == 'BGE').sum())
    n_ties     = int((df['winner'] == 'Tie').sum())
    print(f"Per-image wins:      MedEmbed={n_med_wins}  BGE={n_bge_wins}  Tie={n_ties}")
    print(f"  → More wins:        {'MedEmbed' if n_med_wins > n_bge_wins else ('BGE' if n_bge_wins > n_med_wins else 'Tie')}")

    bge_sep = df['sim_bge'].std()
    med_sep = df['sim_medembed'].std()
    print(f"Std (spread):        BGE={bge_sep:.4f}  vs  MedEmbed={med_sep:.4f}")

    med_score = sum([med_mean > bge_mean,
                     med_match >= bge_match,
                     n_med_wins >= n_bge_wins])
    print(f"\nOverall verdict:     {'MedEmbed' if med_score >= 2 else 'BGE'} "
          f"(MedEmbed wins {med_score}/3 primary criteria)")

    # Save per-image results
    out_cols = [
        'image_name',
        'sim_bge', 'sim_medembed', 'diff_med_minus_bge', 'winner',
        'gt_words', 'gen_words',
        GT_COLUMN, GEN_COLUMN,
    ]
    df[out_cols].to_csv(output_csv, index=False)
    print(f"\nPer-image results saved: {output_csv}")

    # Summary CSV
    summary = pd.DataFrame([
        {
            'specialist': label,
            'model': GENERAL_MODEL,
            'type': 'general-purpose',
            'n': len(df),
            'mean': round(bge_mean, 4),
            'std': round(df['sim_bge'].std(), 4),
            'min': round(df['sim_bge'].min(), 4),
            'max': round(df['sim_bge'].max(), 4),
            'matches_at_0.75': bge_match,
            'match_rate': round(bge_match / len(df), 4),
            'per_image_wins': n_bge_wins,
        },
        {
            'specialist': label,
            'model': MEDICAL_MODEL,
            'type': 'medical-specific',
            'n': len(df),
            'mean': round(med_mean, 4),
            'std': round(df['sim_medembed'].std(), 4),
            'min': round(df['sim_medembed'].min(), 4),
            'max': round(df['sim_medembed'].max(), 4),
            'matches_at_0.75': med_match,
            'match_rate': round(med_match / len(df), 4),
            'per_image_wins': n_med_wins,
        },
    ])
    summary.to_csv(summary_csv, index=False)
    print(f"\nSummary saved: {summary_csv}")

    # Threshold sensitivity
    print(f"\n===== {label.upper()} THRESHOLD SENSITIVITY =====")
    print(f"{'Threshold':>10}  {'BGE match':>12}  {'MedEmbed match':>15}")
    for t in [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
        b = int((df['sim_bge'] >= t).sum())
        m = int((df['sim_medembed'] >= t).sum())
        print(f"{t:>10.2f}  {b:>8}/{len(df):<3}  {m:>11}/{len(df):<3}")

    return df, summary

# ------------------------------------------------------------
# Run both specialists
# ------------------------------------------------------------

pharma_result = run_cosine_eval(
    PHARMA_INPUT_CSV, PHARMA_OUTPUT_CSV, PHARMA_SUMMARY_CSV, "pharma"
)

aria_result = run_cosine_eval(
    ARIA_INPUT_CSV, ARIA_OUTPUT_CSV, ARIA_SUMMARY_CSV, "aria"
)

# ------------------------------------------------------------
# Combined summary (both specialists in one table)
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("COMBINED SUMMARY")
print("=" * 60)

combined_rows = []
for label, res in [("pharma", pharma_result), ("aria", aria_result)]:
    if res is not None:
        combined_rows.append(res[1])
if combined_rows:
    combined = pd.concat(combined_rows, ignore_index=True)
    combined.to_csv('/kaggle/working/mednexus_pharma_aria_cosine_combined.csv', index=False)
    print(combined.to_string(index=False))
    print("\nSaved: /kaggle/working/mednexus_pharma_aria_cosine_combined.csv")

PHARMA — dual-model cosine similarity
Loaded 10 rows from /kaggle/working/mednexus_pharma_extracted_details2.csv
Valid rows: 10

Loading BAAI/bge-large-en-v1.5 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

  mean=0.7753  std=0.0807  min=0.6478  max=0.8911  match@0.75=6/10

===== PHARMA PER-IMAGE COMPARISON =====
                             image_name  sim_bge  sim_medembed  diff_med_minus_bge   winner
          Medical_Report_2026-02-18.png 0.528619      0.647817            0.119198 MedEmbed
      Medical_Report_2026-02-16 (9).png 0.577345      0.690055            0.112710 MedEmbed
     Medical_Report_2026-02-16 (10).png 0.678222      0.765527            0.087305 MedEmbed
Medical_Report_Dr_Zavian_2026-10-24.png 0.701599      0.737175            0.035575 MedEmbed
          Medical_Report_2026-02-12.png 0.662102      0.674023            0.011921 MedEmbed
     Medical_Report_2026-02-16 (11).png 0.807060      0.806885           -0.000175      BGE
     Medical_Report_2026-02-16 (12).png 0.852630      0.842776           -0.009854      BGE
          Medical_Report_2026-02-20.png 0.910898      0.891059           -0.019839      BGE
      Medical_Report_2026-02-15 (5).png 0.880039      0.850213  

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  mean=0.7451  std=0.1451  min=0.4965  max=0.9485  match@0.75=6/10

Loading abhinand/MedEmbed-base-v0.1 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  mean=0.8096  std=0.0794  min=0.6903  max=0.9396  match@0.75=7/10

===== ARIA PER-IMAGE COMPARISON =====
                             image_name  sim_bge  sim_medembed  diff_med_minus_bge   winner
      Medical_Report_2026-02-12 (1).png 0.496544      0.690310            0.193766 MedEmbed
      Medical_Report_2026-02-16 (6).png 0.545153      0.725469            0.180316 MedEmbed
      Medical_Report_2026-02-15 (2).png 0.661475      0.804049            0.142574 MedEmbed
Medical_Report_Dr_Zavian_2026-04-25.png 0.808382      0.884801            0.076419 MedEmbed
Pharmacy_Report_Dr_Phram_2026-04-24.png 0.658810      0.720613            0.061803 MedEmbed
      Medical_Report_2026-02-15 (3).png 0.784972      0.814558            0.029586 MedEmbed
Pharmacy_Report_Dr_Phram_2026-04-25.png 0.772579      0.781785            0.009207 MedEmbed
          Medical_Report_2026-02-16.png 0.832412      0.824153           -0.008259      BGE
          Medical_Report_2026-02-15.png 0.948463      0.939564    